## 0.1 Imports

In [24]:
import pandas as pd
from itertools import combinations
from statsmodels.stats.gof import chisquare_effectsize
from statsmodels.stats.power import GofChisquarePower, TTestIndPower
import numpy as np
import scipy.stats as stats
from statsmodels.sandbox.stats.multicomp import multipletests

# 1.0 Load data

In [2]:
d = {
    'variant': ['interact', 'connect', 'learn', 'help', 'services'],
    'visits': [10283, 2742, 2747, 3180, 2064],
    'clicks_all': [3714, 1587, 1652, 1717, 1348],
    'clicks_link': [42, 53, 21, 38, 45]
}
data = pd.DataFrame(d)
data

,variant,visits,clicks_all,clicks_link
0,interact,10283,3714,42
1,connect,2742,1587,53
2,learn,2747,1652,21
3,help,3180,1717,38
4,services,2064,1348,45


# 2.0 Design de Experimentos

## 2.1 Formulação de Hipóteses

In [ ]:
# H0= Não há nenhuma diferença entre o CTR das variantes da página
# H1= Há uma diferença entre os CTR das variantes da página

# 2.2 Parâmetros de Experimentos

In [3]:
k = len(data['clicks_all'])
actual_dist = data['clicks_link'] / data['clicks_link'].sum()
expected_dist = [1/k]*k

In [ ]:
alpha = 0.05
power = 0.80
effect_size = chisquare_effectsize(expected_dist, actual_dist)




sample_size = TTestIndPower().solve_power (
    effect_size=effect_size,
    alpha = alpha,
    power = power
)

sample_size = np.ceil(sample_size).astype(int)
print('Minimum Sample Size per Variant: {}'.format(sample_size))
print('Total Sample Size: {}'.format(sample_size*k))

# Sample Size
# ---- Effect Size
# ---- Alpha
# ---- Power

Minimum Sample Size per Variant: 222
Total Sample Size: 1110


# 3.0 Aplicação do teste de chi square

In [9]:
data

,variant,visits,clicks_all,clicks_link,no_clicks_link
0,interact,10283,3714,42,3672
1,connect,2742,1587,53,1534
2,learn,2747,1652,21,1631
3,help,3180,1717,38,1679
4,services,2064,1348,45,1303


In [11]:
data['no_clicks_link'] = data['clicks_all'] - data['clicks_link']
df = data[['variant', 'clicks_link', 'no_clicks_link']]
df = df.set_index('variant')

In [17]:
chi2, p_value, dof, ex = stats.chi2_contingency(df)
print('Chi Squared: {} - p-value: {}'.format(chi2,p_value))

Chi Squared: 46.33660181942126 - p-value: 2.0959498129984567e-09


# 4.0 Post-hoc Testings

In [29]:
all_combinatioins = list (combinations(df.index,2))
p_values = []

for comb in all_combinatioins:
    new_df = df[(df.index == comb[0]) | (df.index == comb[1])] 
    chi2, p_value, dof, ex = stats.chi2_contingency(new_df)
    p_values.append(p_value)

# Corretion of Bonferroini
reject_list, corrected_p_values, _, _ = multipletests(p_values, method='bonferroni')

In [30]:
for comb, p_val, corr_p_val, reject in zip(all_combinatioins, p_values, corrected_p_values, reject_list):
    print('\n{}: p_value:{}; corrected_p_value:{}; Reject:{}'.format(comb, p_val, corr_p_val, reject))


('interact', 'connect'): p_value:5.3676772349808135e-08; corrected_p_value:5.367677234980813e-07; Reject:True

('interact', 'learn'): p_value:0.7616980743361713; corrected_p_value:1.0; Reject:False

('interact', 'help'): p_value:0.0031030587017400212; corrected_p_value:0.03103058701740021; Reject:True

('interact', 'services'): p_value:1.798089447385411e-07; corrected_p_value:1.7980894473854111e-06; Reject:True

('connect', 'learn'): p_value:0.00013292868361715983; corrected_p_value:0.0013292868361715984; Reject:True

('connect', 'help'): p_value:0.06144184057612575; corrected_p_value:0.6144184057612575; Reject:False

('connect', 'services'): p_value:1.0; corrected_p_value:1.0; Reject:False

('learn', 'help'): p_value:0.0508958228881819; corrected_p_value:0.5089582288818191; Reject:False

('learn', 'services'): p_value:0.00020374035733741825; corrected_p_value:0.0020374035733741825; Reject:True

('help', 'services'): p_value:0.07301998638337415; corrected_p_value:0.7301998638337415; R

In [ ]:
Interect x Connect -> Há um relação, diferença, dependencia -> p-value = 5.367677234980813e-07
Interect x Services -> Há um relação, diferença, dependencia -> p-value = 1.7980894473854111e-06
Interect x Help -> Há um relação, diferença, dependencia -> p-value = 0.03103058701740021


Connect x Help -> Não há diferença
Connect x Service -> Não há diferença
Help x Services -> Não há diferença